# DOCX → Azota + UniMERNet + Unlimited-OCR

Notebook này hoàn thành pipeline đề thi Word → cú pháp Azota.

**Thứ tự bắt buộc (không OCR-first):**

1. **OOXML walker** (CPU) → `markup.txt` + `sidecar/` + `manifest.json`  
   Giữ `[!b:$…$]`, `[!m:$mathml_N$]`, `[!m:$mathtype_N$]`, `[img:$img_N$]`, bảng `[* … *]`, dấu `*` đáp án đúng.
2. **UniMERNet** ([opendatalab/UniMERNet](https://github.com/opendatalab/UniMERNet)) — crop WMF MathType → LaTeX, ghi `sidecar/mathtype_N.tex`.
3. **Unlimited-OCR** ([baidu/Unlimited-OCR](https://github.com/baidu/Unlimited-OCR), kiến trúc R-SWA trong [Unlimited-OCR.png](https://github.com/baidu/Unlimited-OCR/blob/main/assets/Unlimited-OCR.png)) — parse cả trang để QA hình vẽ / bảng-ảnh / chữ thiếu.

Runtime: **T4** đủ bước 1–2. Bật Unlimited-OCR trên **A100/L4** (mô hình ~3B).


In [ ]:
#@title Cấu hình
REPO_URL = "https://github.com/phuchoang2603/refurbished-marketplace.git"
REPO_BRANCH = "cursor/docx-to-azota-pipeline-4d56"  # đổi thành main sau khi merge
RUN_UNIMERNET = True
UNIMERNET_SIZE = "tiny"  # tiny | small | base
RUN_UNLIMITED_OCR = False  # True trên A100
OCR_GUNDAM = True  # crop_mode cho 1 trang; multi-page luôn dùng base
OUT_DIR = "/content/azota_out"
print("GPU:")
!nvidia-smi -L || echo "CPU only"


## 0. Cài converter + (tuỳ chọn) vision deps


In [ ]:
import sys, shutil
from pathlib import Path

if not Path("/content/docx-to-azota").exists():
    !git clone -b {REPO_BRANCH} --depth 1 {REPO_URL} /content/refurbished-marketplace
    src = Path("/content/refurbished-marketplace/tools/docx-to-azota")
    if src.exists():
        shutil.copytree(src, "/content/docx-to-azota")
    else:
        raise FileNotFoundError("Không thấy tools/docx-to-azota — upload folder đó lên /content/docx-to-azota")

sys.path.insert(0, "/content/docx-to-azota")
from convert import convert_docx, apply_unimernet_latex, write_ocr_sidecar
print("converter OK")


In [ ]:
#@title Cài GPU packages (chỉ khi bật overlay)
if RUN_UNIMERNET:
    !pip -q install -U "unimernet[full]" pillow pymupdf
    !apt-get -qq install -y imagemagick libmagickwand-dev >/dev/null
    !pip -q install Wand || true
if RUN_UNLIMITED_OCR:
    !pip -q install -U transformers==4.57.1 einops addict easydict pymupdf
    !apt-get -qq install -y libreoffice >/dev/null
print("deps ready")


## 1. Upload `.docx` (hoặc dùng đề mẫu)


In [ ]:
from google.colab import files

uploaded = files.upload()  # chọn ĐỀ VẬT LÍ LẦN 3_VER 2 (2).docx
if uploaded:
    DOCX_PATH = "/content/" + next(iter(uploaded))
else:
    DOCX_PATH = "/content/docx-to-azota/samples/de-vat-li-lan-3.docx"
print("DOCX:", DOCX_PATH)


## 2. Bước A — OOXML → Azota (nguồn sự thật)


In [ ]:
import json
from pathlib import Path

manifest = convert_docx(DOCX_PATH, OUT_DIR)
print(json.dumps(manifest["counts"], ensure_ascii=False, indent=2))
print("--- markup.txt (80 dòng đầu) ---")
print("\n".join(Path(OUT_DIR, "markup.txt").read_text(encoding="utf-8").splitlines()[:80]))


## 3. Bước B — UniMERNet: MathType WMF → LaTeX

MathType OLE (`Equation.DSMT4`) không chứa LaTeX trong XML. UniMERNet đọc ảnh preview (WMF/EMF đã copy vào `sidecar/mathtype_N.wmf`) và ghi LaTeX cạnh placeholder `[!m:$mathtype_N$]` — **không thay placeholder** (Azota vẫn trỏ sidecar gốc).


In [ ]:
from vision import rasterize_formula_image, run_unimernet
from convert import apply_unimernet_latex

if RUN_UNIMERNET:
    man = json.loads(Path(OUT_DIR, "manifest.json").read_text(encoding="utf-8"))
    jobs = []
    png_dir = Path(OUT_DIR) / "sidecar_png"
    png_dir.mkdir(exist_ok=True)
    for asset in man["assets"]:
        if asset["kind"] != "mathtype":
            continue
        src = Path(OUT_DIR) / asset["sidecar"]
        if not src.exists():
            continue
        dest = png_dir / f"{asset['id']}.png"
        got = rasterize_formula_image(src, dest)
        if got:
            jobs.append((asset["id"], got))
    print(f"{len(jobs)} formula images → UniMERNet ({UNIMERNET_SIZE})")
    if jobs:
        import unimernet
        pkg = Path(unimernet.__file__).resolve().parent
        if not (pkg / "configs" / "demo.yaml").exists() and not Path("/content/UniMERNet").exists():
            !git clone --depth 1 https://github.com/opendatalab/UniMERNet.git /content/UniMERNet
            !mkdir -p /content/UniMERNet/models
            !git lfs install
            ckpt = {"tiny": "unimernet_tiny", "small": "unimernet_small", "base": "unimernet_base"}[UNIMERNET_SIZE]
            !git clone --depth 1 https://huggingface.co/wanderkid/{ckpt} /content/UniMERNet/models/{ckpt}
        cfg = "/content/UniMERNet/configs/demo.yaml" if Path("/content/UniMERNet/configs/demo.yaml").exists() else None
        preds = run_unimernet(jobs, model_size=UNIMERNET_SIZE, cfg_path=cfg)
        apply_unimernet_latex(man, preds, Path(OUT_DIR))
        for k, v in list(preds.items())[:8]:
            print(k, "→", v[:120])
else:
    print("SKIP UniMERNet")


## 4. Bước C — Unlimited-OCR: parse cả trang (QA / chữ trong hình)

Unlimited-OCR thay attention decoder bằng **Reference Sliding Window Attention (R-SWA)** nên KV cache không phình khi đề dài — đúng bài toán *one-shot long-horizon parsing* (PDF nhiều trang, hình + chữ lẫn).

Dùng **gundam** (`base_size=1024, image_size=640, crop_mode=True`) cho từng trang chi tiết; **base** + `infer_multi` khi ghép nhiều trang.

Output OCR **không ghi đè** `markup.txt`. Nó nằm ở `ocr/unlimited_ocr.md` để đối chiếu hình vẽ / bảng-ảnh.


In [ ]:
from vision import pdf_to_images, run_unlimited_ocr_pages, strip_unlimited_ocr_det

if RUN_UNLIMITED_OCR:
    !soffice --headless --convert-to pdf --outdir /content "{DOCX_PATH}"
    produced = list(Path("/content").glob("*.pdf"))
    if not produced:
        raise RuntimeError("LibreOffice không tạo được PDF")
    pages = pdf_to_images(str(produced[0]), dpi=200)
    print(f"{len(pages)} pages")
    raw = run_unlimited_ocr_pages(
        pages,
        output_path=f"{OUT_DIR}/ocr_raw",
        gundam=OCR_GUNDAM and len(pages) == 1,
    )
    cleaned = strip_unlimited_ocr_det(raw)
    write_ocr_sidecar(cleaned, Path(OUT_DIR))
    print(cleaned[:2000])
else:
    print("SKIP Unlimited-OCR — markup.txt từ OOXML đã đủ import Azota.")
    print("Bật RUN_UNLIMITED_OCR=True nếu cần đọc chữ trong hình thí nghiệm.")


## 5. QA nhanh + tải kết quả

Kiểm tra: mỗi placeholder trong `markup.txt` có file sidecar; mỗi `*A.`–`*D.` / `*a)` khớp đáp án.


In [ ]:
import re, json
from pathlib import Path
from google.colab import files

out = Path(OUT_DIR)
text = (out / "markup.txt").read_text(encoding="utf-8")
man = json.loads((out / "manifest.json").read_text(encoding="utf-8"))
missing = []
for a in man["assets"]:
    p = out / a["sidecar"]
    if not p.exists() or p.stat().st_size == 0:
        missing.append(a["id"])
print("assets", man["counts"], "missing sidecar", missing)
print("MCQ stars", re.findall(r"\*[A-D]\.", text))
print("TF stars", re.findall(r"\*[a-d]\)", text))
print("short answers", re.findall(r"→ Đáp án:.*", text))

!cd /content && zip -qr azota_out.zip azota_out
files.download("/content/azota_out.zip")
